In [1]:
import requests
import json
import gzip
from datetime import datetime, timedelta

# TMDB cập nhật file ID mỗi ngày, ta sẽ lấy file của ngày hôm qua
yesterday = (datetime.now() - timedelta(1)).strftime('%m_%d_%Y')
url = f"http://files.tmdb.org/p/exports/movie_ids_{yesterday}.json.gz"

print("Đang tải danh sách Movie IDs...")
response = requests.get(url)

with open('movie_ids.json.gz', 'wb') as f:
    f.write(response.content)

# Đọc file .gz và trích xuất danh sách ID
movie_ids = []
with gzip.open('data/movie_ids.json.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        movie_ids.append(data['id'])

print(f"Tổng số phim trên TMDB: {len(movie_ids)}")

Đang tải danh sách Movie IDs...
Tổng số phim trên TMDB: 1172698


In [3]:
import requests
import pandas as pd
import time
import os

API_KEY = "aaf5fb3d804913077429c21cafbf41ac"
OUTPUT_FILE = "data/tmdb_implicit_feedback.csv"

# Khởi tạo file CSV nếu chưa có
if not os.path.exists(OUTPUT_FILE):
    df_empty = pd.DataFrame(columns=['user_id', 'item_id', 'interaction', 'timestamp'])
    df_empty.to_csv(OUTPUT_FILE, index=False)

collected_records = 0
batch_data = []

print("Bắt đầu cào dữ liệu reviews...")
# Quét qua từng movie_id (giả sử bạn lấy 100,000 ID đầu tiên để chạy thử)
for idx, movie_id in enumerate(movie_ids[:100000]):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?api_key={API_KEY}"
    
    try:
        response = requests.get(url, timeout=10)
        
        # Xử lý Rate Limit (nếu quá số lượng request, TMDB trả về code 429)
        if response.status_code == 429:
            print("Chạm giới hạn API! Tạm nghỉ 5 giây...")
            time.sleep(5)
            continue
            
        if response.status_code == 200:
            reviews = response.json().get('results', [])
            for rev in reviews:
                batch_data.append({
                    'user_id': rev['author'],
                    'item_id': movie_id,
                    'interaction': 1, # Đưa về implicit feedback = 1
                    'timestamp': rev['created_at']
                })
                collected_records += 1
                
        # Cứ mỗi 100 phim, lưu dữ liệu xuống ổ cứng 1 lần (Làm sạch và chuẩn hóa bước đầu)
        if (idx + 1) % 100 == 0:
            if batch_data:
                df_batch = pd.DataFrame(batch_data)
                # Lưu append vào file CSV
                df_batch.to_csv(OUTPUT_FILE, mode='a', header=False, index=False)
                batch_data = [] # Reset batch
            print(f"Đã quét {idx + 1} phim | Tổng số tương tác thu được: {collected_records}")
            
            # Nghỉ 0.1s mỗi 100 request để an toàn cho API quota
            time.sleep(0.1)

    except Exception as e:
        print(f"Lỗi ở phim ID {movie_id}: {e}")
        continue

print(f"Hoàn thành! Tổng số dòng dữ liệu cào được: {collected_records}")

Bắt đầu cào dữ liệu reviews...


KeyboardInterrupt: 